In [7]:
import sys
from pathlib import Path
path = Path.cwd().parents[1]/"app"
if str(path) not in sys.path:
    sys.path.insert(0, str(path))
from models.database import MBSE
import pandas as pd
from sqlalchemy import inspect
from models import mbse

def sqlalchemy_schema_to_df(base, include_constraints=True,):
    """Convert SQLAlchemy declarative metadata into a DataFrame."""
    metadata = base.metadata
    rows = []
    for table in metadata.sorted_tables:
        # Identify foreign keys by column.
        foreign_keys = {fk.parent.name: (f"{fk.target_fullname}") for fk in table.foreign_keys}
        # Identify columns belonging to unique constraints.
        unique_columns = set()
        for constraint in table.constraints:
            if constraint.__class__.__name__ == "UniqueConstraint":
                unique_columns.update(column.name for column in constraint.columns)
        # Identify indexed columns.
        indexed_columns = set()
        for index in table.indexes:
            indexed_columns.update(column.name for column in index.columns)
        for column in table.columns:
            rows.append({
                "schema": table.schema,
                "table": table.name,
                "column": column.name,
                "type": str(column.type),
                "nullable": column.nullable,
                "primary_key": column.primary_key,
                "foreign_key": foreign_keys.get(column.name),
                "unique": column.name in unique_columns,
                "indexed": column.name in indexed_columns,
                "default": (str(column.default.arg) if column.default is not None else None),
                "server_default": str(column.server_default.arg) if column.server_default is not None else None
            })
    return pd.DataFrame(rows)

df_schema = sqlalchemy_schema_to_df(MBSE)
print(df_schema.to_string(index=False))

schema                            table                       column       type  nullable  primary_key                  foreign_key  unique  indexed                                             default server_default
  mbse                engineering_model                         name       TEXT     False        False                         None    True    False                                                None           None
  mbse                engineering_model                  description       TEXT      True        False                         None   False    False                                                None           None
  mbse                engineering_model                  source_tool       TEXT      True        False                         None   False    False                                                None           None
  mbse                engineering_model                source_format       TEXT      True        False                         None   Fa